# 1. Whole-tumor segmentation with a fine-tuned MedSAM2

Reproduces **Section 3.1** (the method) and **Section 4.1** (the results), namely:

| output | paper |
|---|---|
| whole-tumor Dice, IoU, precision, recall with the worst-5% column | `tab:seg` |
| held-out Dice before and after fine-tuning | Section 4.1, "from about 0.76 to 0.86" |
| the two fine-tuning inputs reaching the same held-out Dice | Section 4.1 and the Conclusions |
| how much of each reference sub-region survives inside the predicted mask | Section 4.1, 94.8% / 97.1% / 88.9% |

**This notebook needs the BraTS working copy.** The fine-tune takes roughly 55 minutes for six
epochs on an Apple GPU and the volumes are several gigabytes, so the training and inference cells
are the only ones in this release that cannot be run from the shipped caches.

### Which masks the rest of the pipeline uses

Three FLAIR fine-tunes exist in the history of this work and only one of them produced the masks
every later notebook consumes:

| run | epochs | best held-out slice Dice | what it is |
|---|---|---|---|
| FLAIR, twelve epochs | 12 | 0.8633 | produced the predicted masks in `brats_seg_ft/`, which every downstream table reads |
| FLAIR, six epochs | 6 | 0.8642 | the run the 0.76 to 0.86 figure comes from |
| FLAIR with T1ce and T2, six epochs | 6 | 0.8644 | the second input method, reported as equivalent |

The recipe below is the six-epoch one, and it is what this notebook reproduces. It does not
reproduce the twelve-epoch checkpoint bit for bit, so the masks it writes go to their own
directory and never overwrite `brats_seg_ft/`. Everything reported in the paper from those masks
is therefore reproduced from the recipe, not from a rebuilt checkpoint.

Two further points the paper states as limitations and this notebook makes visible:

- the box prompts come from the reference masks, so this stage still needs an annotation;
- the cohort Dice of 0.913 is measured on cases the model was fine-tuned on, which is why the
  held-out 0.86 is the number to read as performance on unseen data.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import medsam2, segmentation, cohort

medsam2.hyperparameters()

## Normalization is already applied

The stored volumes have been clipped to their 1st and 99th percentiles and scaled to 8-bit
already. Applying the same transform a second time would move about 15% of voxels by one level
and the input would no longer be the one the published fine-tune saw, so the arrays are used as
they are. The cell below shows this instead of assuming it: the percentiles sit at 0 and 255, and
re-applying the transform changes nothing beyond re-rounding.

In [ ]:
medsam2.normalization_proof(medsam2.CHANNEL_SETS["multichannel"])

## Cohort split and cost

The split is at case level, so no slice of a validation case is seen during training. The timing
cell measures seconds per iteration on this machine; on an Apple GPU it came out at about
0.169 s/it, which is roughly 55 minutes for six epochs.

In [ ]:
train_files, val_files = medsam2.case_split()
print(f"{len(train_files)} training cases, {len(val_files)} validation cases")

# Uncomment to measure this machine's throughput before committing to a full run.
# medsam2.timed_batches(medsam2.CHANNEL_SETS["flair"])

## Zero-shot baseline, before any fine-tuning

MedSAM2 out of the box, prompted with the same boxes, reaches a mean Dice of **0.7694**. That is
the 0.76 the paper compares against.

In [ ]:
RUN_HEAVY = False   # set True with the BraTS copy in place

if RUN_HEAVY:
    channels = medsam2.CHANNEL_SETS["flair"]
    zero_shot_dir = config.DATA_ROOT / "brats_seg_zeroshot"
    medsam2.segment_all(channels, seg_out=zero_shot_dir,
                        model=medsam2.build_model("eval"))
    ids, metrics, _, _ = segmentation.scan_masks({"zero-shot": zero_shot_dir})
    print(f"zero-shot mean Dice over {len(ids)} cases: {metrics['zero-shot'][:, 0].mean():.4f}")

## Fine-tune, both inputs

`flair` replicates one channel three times, which is what the published run did.
`multichannel` puts FLAIR, T1ce and T2 in the three channels. Everything else is identical, so
the comparison isolates the input.

Expected best held-out slice Dice: **0.8642** for FLAIR and **0.8644** for the three-channel
input. The paper reports both as 0.86 and calls them equivalent.

In [ ]:
if RUN_HEAVY:
    for name in ("flair", "multichannel"):
        history, best = medsam2.fine_tune(medsam2.CHANNEL_SETS[name])
        display(history)
        print(f"{name}: best held-out slice Dice {best:.4f}")

In [ ]:
if RUN_HEAVY:
    for name in ("flair", "multichannel"):
        out = medsam2.segment_all(medsam2.CHANNEL_SETS[name])
        assert medsam2.check_masks(out), f"mask integrity failed for {name}"
        print(f"{name} masks in {out}")

## tab:seg -- the masks the rest of the pipeline works from

Scored per case against the reference, over the 368 cases that carry one. Expected:

| Metric | Mean | Std | Worst 5% |
|---|---|---|---|
| Dice | 0.913 | 0.035 | 0.836 |
| IoU | 0.842 | 0.057 | 0.719 |
| Precision | 0.908 | 0.039 | 0.828 |
| Recall | 0.919 | 0.042 | 0.844 |

with Dice 0.911 on the high-grade cases and 0.920 on the low-grade ones. Precision and recall
sit close together, so the prediction is not biased towards over- or under-segmenting.

In [ ]:
if RUN_HEAVY:
    ids, metrics, recovery, pooled = segmentation.scan_masks({"predicted": config.SEG_DIR})
    grades = cohort.load_grade_labels()
    grade = np.array([grades.get(c, "?") for c in ids])
    print(f"n = {len(ids)}  "
          f"(high-grade {(grade == 'HGG').sum()}, low-grade {(grade == 'LGG').sum()})")
    display(segmentation.segmentation_table(metrics, "predicted", grade=grade))
    display(segmentation.worst_cases(ids, metrics, "predicted", grade=grade))

## Where the missing tissue is

The errors are not spread evenly. The predicted mask holds **94.8%** of the reference enhancing
tumor and **97.1%** of the necrotic core, but only **88.9%** of the edema. Almost everything the
segmentation misses is edema at the outer margin, and the enhancing tumor and necrotic core,
which is where the grade is read from, are nearly always inside the mask. That is the first of
the two reasons the substitution in notebook 04 changes so little.

The weakest case in the cohort, at a Dice of 0.74, still keeps 99% of the enhancing tumor and
necrotic core while missing 41% of the edema.

In [ ]:
if RUN_HEAVY:
    display(segmentation.recovery_table(recovery, pooled, ["predicted"]))
    print(segmentation.case_recovery("BraTS20_Training_048"))

## The two inputs side by side

Both fine-tunes reach the same held-out Dice, the cohort-level metrics differ in the third
decimal, and grading from either set of masks gives 0.93. The numbers reported in the paper come
from the first method.

In [ ]:
if RUN_HEAVY:
    _, mc_ckpt, mc_dir, mc_cache = medsam2.run_paths(medsam2.CHANNEL_SETS["multichannel"])
    keys = ["FLAIR", "FLAIR, T1ce, T2"]
    ids, metrics, recovery, pooled = segmentation.scan_masks(
        {keys[0]: config.SEG_DIR, keys[1]: mc_dir})
    display(segmentation.segmentation_table(metrics, keys[0]))
    display(segmentation.segmentation_table(metrics, keys[1]))
    display(segmentation.recovery_table(recovery, pooled, keys))

In [ ]:
if RUN_HEAVY:
    from src import tables
    grades = cohort.load_grade_labels()
    X_flair, y, cases_flair = cohort.build_matrix(cohort.extract_features(), grades)
    feat_mc = cohort.extract_features(seg_dir=mc_dir, cache=mc_cache)
    X_mc, y_mc, cases_mc = cohort.build_matrix(feat_mc, grades)
    assert cases_flair == cases_mc, "case order must match or the folds differ"

    side_by_side = pd.DataFrame({
        "FLAIR masks": tables.table_clf(X_flair, y)["AUC"],
        "FLAIR, T1ce, T2 masks": tables.table_clf(X_mc, y_mc)["AUC"],
    })
    side_by_side["difference"] = (side_by_side.iloc[:, 1] - side_by_side.iloc[:, 0]).round(4)
    display(side_by_side)